In [1]:
!pip install numpy scikit-learn

In [33]:
import json
import numpy as np
import re
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

In [35]:
with open("student_performance.json", encoding="utf-8") as f:
    students = json.load(f)

with open("question_bank.json", encoding="utf-8") as f:
    question_bank = json.load(f)

with open("dost_config.json", encoding="utf-8") as f:
    dost_config = json.load(f)

print("Data loaded successfully ✅")

Data loaded successfully ✅


In [37]:
import re

def normalize_marks(marks):
    # Handle integer marks
    if isinstance(marks, int):
        return marks

    # Handle string formats
    if isinstance(marks, str):
        marks = marks.strip()

        # Case 1: "68/100"
        if "/" in marks:
            try:
                num = float(marks.split("/")[0])
                denom = float(re.findall(r'/(\d+)', marks)[0])
                return (num / denom) * 100
            except:
                return 0

        # Case 2: "+52 -8"
        if "+" in marks and "-" in marks:
            try:
                parts = marks.split()
                pos = int(parts[0].replace("+", ""))
                neg = int(parts[1].replace("-", ""))
                return pos - neg
            except:
                return 0

        # Case 3: "45" or "22"
        try:
            return float(marks)
        except:
            return 0

    return 0


def clean_questions(questions):
    cleaned = []
    seen = set()

    for q in questions:
        # Handle _id format
        qid = q.get("_id")

        if isinstance(qid, dict):
            qid = qid.get("$oid")

        if not qid or qid in seen:
            continue

        # Fix difficulty
        difficulty = q.get("difficulty")
        if difficulty is None:
            difficulty = 3

        # Extract answer
        answer = None

        if q.get("scq"):
            answer = q["scq"].get("answer")
        elif q.get("mcq"):
            answer = q["mcq"].get("answer")
        elif q.get("integerQuestion"):
            answer = q["integerQuestion"].get("answer")

        # Skip bad questions
        if answer is None:
            continue

        # Add cleaned fields
        q["clean_id"] = qid
        q["difficulty"] = difficulty

        cleaned.append(q)
        seen.add(qid)

    return cleaned

In [39]:
clean_qs = clean_questions(question_bank)
print("Cleaned questions:", len(clean_qs))

Cleaned questions: 190


In [41]:
def analyze_student(attempts):
    scores = []
    total_attempted = 0
    total_questions = 0
    total_time = 0
    completed_count = 0

    chapter_perf = {}

    for att in attempts:
        score = normalize_marks(att["marks"])
        scores.append(score)

        total_attempted += att["attempted"]
        total_questions += att["total_questions"]
        total_time += att["avg_time_per_question_seconds"]

        if att["completed"]:
            completed_count += 1

        for ch in att["chapters"]:
            chapter_perf.setdefault(ch, []).append(score)

    avg_score = sum(scores) / len(scores)
    accuracy = total_attempted / total_questions
    avg_time = total_time / len(attempts)
    completion_rate = completed_count / len(attempts)

    strengths = []
    weaknesses = []

    for ch, vals in chapter_perf.items():
        avg = sum(vals) / len(vals)
        if avg > 60:
            strengths.append(ch)
        elif avg < 40:
            weaknesses.append(ch)

    issues = []
    if avg_score < 40:
        issues.append("low_score")
    if avg_time > 120:
        issues.append("slow_speed")
    if accuracy < 0.7:
        issues.append("low_accuracy")
    if completion_rate < 0.7:
        issues.append("time_management")

    return {
        "avg_score": round(avg_score, 2),
        "accuracy": round(accuracy, 2),
        "avg_time": round(avg_time, 2),
        "completion_rate": round(completion_rate, 2),
        "strengths": strengths,
        "weaknesses": weaknesses,
        "issues": issues
    }

In [43]:
def build_weakness_profile(analysis):
    profile = {}
    
    for topic in analysis["weaknesses"]:
        profile[topic.lower()] = 1.0
    
    return profile

In [45]:
TOPICS = [
    "mechanics", "thermodynamics", "electrostatics", "optics",
    "modern_physics", "organic_chemistry", "inorganic_chemistry",
    "physical_chemistry", "algebra", "calculus", "coordinate_geometry",
    "trigonometry"
]

TOPIC_TO_IDX = {t: i for i, t in enumerate(TOPICS)}

def build_matrix(records, record_type="student"):
    matrix = np.zeros((len(records), len(TOPICS)))

    if record_type == "student":
        for i, rec in enumerate(records):
            for topic, score in rec.items():
                if topic in TOPIC_TO_IDX:
                    matrix[i, TOPIC_TO_IDX[topic]] = score
    else:
        for i, rec in enumerate(records):
            topic = rec.get("topic", "").lower()
            difficulty = rec.get("difficulty", 3)

            if topic in TOPIC_TO_IDX:
                matrix[i, TOPIC_TO_IDX[topic]] = difficulty

    return normalize(matrix)

In [47]:
def recommend_questions(student_matrix, question_matrix, questions, idx, top_n=10):
    cohort_baseline = student_matrix.mean(axis=0)

    # FIXED logic
    student_profile = student_matrix[idx] - cohort_baseline
    student_profile = np.maximum(student_profile, 0)

    norm = np.linalg.norm(student_profile)
    student_profile = student_profile / (norm + 1e-10)

    sims = cosine_similarity(student_profile.reshape(1, -1), question_matrix).flatten()

    top_idx = np.argsort(sims)[::-1][:top_n]

    return [
        {
            "qid": questions[i]["clean_id"],
            "topic": questions[i]["topic"],
            "difficulty": questions[i]["difficulty"],
            "score": round(float(sims[i]), 4)
        }
        for i in top_idx
    ]

In [49]:
all_students_profiles = []
analyses = []

for s in students:
    analysis = analyze_student(s["attempts"])
    analyses.append(analysis)

    profile = build_weakness_profile(analysis)
    all_students_profiles.append(profile)

student_matrix = build_matrix(all_students_profiles, "student")
question_matrix = build_matrix(clean_qs, "question")

In [55]:
def print_full_report(student_name, analysis, recs):
    print("\n" + "="*70)
    print(f"🎓 STUDENT REPORT: {student_name.upper()}")
    print("="*70)

    print("\n📊 PERFORMANCE SUMMARY")
    print("-"*70)
    print(f"⭐ Average Score        : {analysis['avg_score']}")
    print(f"🎯 Accuracy            : {analysis['accuracy']*100:.1f}%")
    print(f"⏱ Avg Time/Question   : {analysis['avg_time']} sec")
    print(f"📌 Completion Rate     : {analysis['completion_rate']*100:.1f}%")

    print("\n🧠 PERFORMANCE INSIGHTS")
    print("-"*70)

    if analysis["avg_score"] < 40:
        print("❌ Overall performance is LOW — needs concept improvement")
    elif analysis["avg_score"] < 70:
        print("⚠️ Performance is AVERAGE — needs practice")
    else:
        print("✅ Performance is GOOD — keep improving")

    if analysis["avg_time"] > 120:
        print("🐢 You are solving questions SLOWLY")
    else:
        print("⚡ Good solving speed")

    if analysis["accuracy"] < 0.7:
        print("🎯 Accuracy is LOW — focus on correctness")
    else:
        print("✅ Accuracy is GOOD")

    print("\n💪 STRENGTH AREAS")
    print("-"*70)
    print(", ".join(analysis["strengths"]) if analysis["strengths"] else "None")

    print("\n📉 WEAK AREAS")
    print("-"*70)
    print(", ".join(analysis["weaknesses"]) if analysis["weaknesses"] else "None")

    print("\n⚠️ ISSUES DETECTED")
    print("-"*70)
    if analysis["issues"]:
        for issue in analysis["issues"]:
            print(f"🚨 {issue.replace('_',' ').title()}")
    else:
        print("No major issues 🎉")

    print("\n🎯 TOP RECOMMENDATIONS")
    print("-"*70)
    for i, r in enumerate(recs[:5], 1):
        print(f"{i}. {r['qid']} | {r['topic']} | diff={r['difficulty']} | score={r['score']}")

In [57]:
for i, s in enumerate(students):
    analysis = analyses[i]
    recs = recommend_questions(student_matrix, question_matrix, clean_qs, i)

    print_full_report(s["name"], analysis, recs)


🎓 STUDENT REPORT: ARJUN SHARMA

📊 PERFORMANCE SUMMARY
----------------------------------------------------------------------
⭐ Average Score        : 34.14
🎯 Accuracy            : 77.0%
⏱ Avg Time/Question   : 187.67 sec
📌 Completion Rate     : 67.0%

🧠 PERFORMANCE INSIGHTS
----------------------------------------------------------------------
❌ Overall performance is LOW — needs concept improvement
🐢 You are solving questions SLOWLY
✅ Accuracy is GOOD

💪 STRENGTH AREAS
----------------------------------------------------------------------
None

📉 WEAK AREAS
----------------------------------------------------------------------
Thermodynamics, Kinematics

⚠️ ISSUES DETECTED
----------------------------------------------------------------------
🚨 Low Score
🚨 Slow Speed
🚨 Time Management

🎯 TOP RECOMMENDATIONS
----------------------------------------------------------------------
1. 3209eb83425ded302b2ac09d | thermodynamics | diff=4 | score=1.0
2. c80621db212f19d54dbcecc2 | thermodynami

In [59]:
import os
os.makedirs("sample_outputs", exist_ok=True)

for i, s in enumerate(students):
    recs = recommend_questions(student_matrix, question_matrix, clean_qs, i)

    data = {
        "student": s["name"],
        "analysis": analyses[i],
        "recommendations": recs
    }

    filename = f"sample_outputs/{s['name']}.json"

    with open(filename, "w") as f:
        json.dump(data, f, indent=4)

print("✅ All outputs saved in sample_outputs folder!")

✅ All outputs saved in sample_outputs folder!
